# Recuperação de Tickers Históricos via Macrotrends / Stooq

Coleta dados dos 34 tickers pendentes no `ausencias_report.md`:
- 31 irrecuperáveis via Yahoo/Tiingo
- 3 parciais (FOX, FOXA, IR)

**Fluxo:** macrotrends (scraping) → stooq (CSV direto) → falha documentada  
**Saída:** `data_bases/external/macrotrends_recovery/{TICKER}.csv` (não sobrescreve prices/ diretamente)  
**Formato:** `Date,{TICKER}` — igual ao padrão de `data_bases/prices/`

In [18]:
import requests
import re
import json
import time
import shutil
import pandas as pd
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
PRICES_DIR    = PROJECT_ROOT / "data_bases" / "prices"
RECOVERY_DIR  = PROJECT_ROOT / "data_bases" / "external" / "macrotrends_recovery"
RECOVERY_DIR.mkdir(parents=True, exist_ok=True)

print("Prices dir :", PRICES_DIR)
print("Recovery dir:", RECOVERY_DIR)

Prices dir : C:\Users\jvlei\Desktop\TCC-pair-trading\new_aproach\data_bases\prices
Recovery dir: C:\Users\jvlei\Desktop\TCC-pair-trading\new_aproach\data_bases\external\macrotrends_recovery


In [19]:
import re
import json
import time
import pandas as pd
from io import StringIO
from pathlib import Path

# pip install cloudscraper
import cloudscraper

SCRAPER = cloudscraper.create_scraper(
    browser={"browser": "chrome", "platform": "windows", "mobile": False}
)


def fetch_macrotrends(mt_ticker: str, slug: str, start: str, end: str) -> pd.DataFrame:
    url = f"https://www.macrotrends.net/stocks/charts/{mt_ticker}/{slug}/stock-price-history"
    try:
        r = SCRAPER.get(url, timeout=30)
        print(f"    macrotrends status: {r.status_code}")
    except Exception as e:
        print(f"    macrotrends error: {e}")
        return pd.DataFrame()

    match = re.search(r'var originalData\s*=\s*(\[.*?\]);', r.text, re.DOTALL)
    if not match:
        # Mostrar trecho do HTML para diagnóstico
        snippet = r.text[:300].replace('\n', ' ')
        print(f"    var originalData não encontrada. HTML: {snippet!r}")
        return pd.DataFrame()

    try:
        raw = json.loads(match.group(1))
    except json.JSONDecodeError as e:
        print(f"    JSON parse error: {e}")
        return pd.DataFrame()

    if not raw:
        print(f"    Array vazio")
        return pd.DataFrame()

    df = pd.DataFrame(raw)
    df.columns = [c.lower() for c in df.columns]

    close_col = next((c for c in ["close", "adjclose", "adj_close"] if c in df.columns), None)
    if close_col is None or "date" not in df.columns:
        print(f"    Colunas inesperadas: {list(df.columns)}")
        return pd.DataFrame()

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date", close_col])
    df = df[(df["date"] >= start) & (df["date"] <= end)]
    df = df[["date", close_col]].copy()
    df.columns = ["Date", "close"]
    return df.sort_values("Date").reset_index(drop=True)


# --- Teste rápido com 2 tickers antes de rodar tudo ---
print("=== TESTE MACROTRENDS + CLOUDSCRAPER ===\n")
for mt_ticker, slug in [("MON", "monsanto"), ("META", "meta-platforms")]:
    print(f"[{mt_ticker}/{slug}]")
    df = fetch_macrotrends(mt_ticker, slug, "2016-01-01", "2018-06-07")
    if not df.empty:
        print(f"  ✅ {len(df)} linhas | {df['Date'].min().date()} → {df['Date'].max().date()}")
        print(df.head(3).to_string(index=False))
    print()

=== TESTE MACROTRENDS + CLOUDSCRAPER ===

[MON/monsanto]
    macrotrends status: 403
    var originalData não encontrada. HTML: '<!DOCTYPE html><html lang="en-US"><head><title>Just a moment...</title><meta http-equiv="Content-Type" content="text/html; charset=UTF-8"><meta http-equiv="X-UA-Compatible" content="IE=Edge"><meta name="robots" content="noindex,nofollow"><meta name="viewport" content="width=device-width,initial-scal'

[META/meta-platforms]
    macrotrends status: 403
    var originalData não encontrada. HTML: '<!DOCTYPE html><html lang="en-US"><head><title>Just a moment...</title><meta http-equiv="Content-Type" content="text/html; charset=UTF-8"><meta http-equiv="X-UA-Compatible" content="IE=Edge"><meta name="robots" content="noindex,nofollow"><meta name="viewport" content="width=device-width,initial-scal'



In [5]:
# Formato: (orig_ticker, mt_ticker, mt_slug, start, end)
# - orig_ticker : nome do arquivo CSV de saída (ex: 'FB.csv')
# - mt_ticker   : ticker usado pela macrotrends na URL (pode ser o sucessor)
# - mt_slug     : slug da URL macrotrends
# - start/end   : janela de datas necessária

targets = [
    # ==================================================================
    # GRUPO 1 — Ticker reciclado: sucessor com ticker diferente
    # Macrotrends mantém histórico pré-rename sob o ticker atual
    # ==================================================================
    ("FB",    "META",  "meta-platforms",             "2016-01-01", "2021-10-28"),
    ("LB",    "BBWI",  "bath-body-works",            "2016-01-01", "2021-08-02"),
    ("ANTM",  "ELV",   "elevance-health",            "2016-01-01", "2022-06-28"),
    ("STI",   "TFC",   "truist-financial",           "2016-01-01", "2019-12-06"),
    ("VIAC",  "PARA",  "paramount-global",           "2019-12-05", "2022-02-15"),
    ("FBHS",  "FBIN",  "fortune-brands-innovations", "2016-01-01", "2022-11-08"),
    ("TMK",   "GL",    "globe-life",                 "2016-01-01", "2019-08-08"),
    ("DISCA", "WBD",   "warner-bros-discovery",      "2016-01-01", "2022-02-15"),
    ("PKI",   "RVTY",  "revvity",                    "2016-01-01", "2023-05-04"),
    # HCP renomeou para PEAK em 2019 — mesma empresa, página PEAK cobre histórico completo
    ("HCP",   "PEAK",  "healthpeak-properties",      "2016-01-01", "2019-06-30"),
    # ARNC original → split em HWM (Howmet) + novo ARNC em abr/2020
    # HWM é o sucessor legal do ARNC original
    ("ARNC",  "HWM",   "howmet-aerospace",           "2016-11-01", "2020-04-01"),

    # ==================================================================
    # GRUPO 2 — Empresa adquirida/dissolvida: página histórica própria
    # Macrotrends mantém páginas de empresas extintas
    # ==================================================================
    ("MON",   "MON",   "monsanto",                   "2016-01-01", "2018-06-07"),
    ("SE",    "SE",    "spectra-energy",             "2016-01-01", "2017-02-27"),
    ("TE",    "TE",    "teco-energy",                "2016-01-01", "2016-07-10"),
    ("CA",    "CA",    "ca-technologies",            "2016-01-01", "2018-11-05"),
    ("EMC",   "EMC",   "emc",                        "2016-01-01", "2016-09-10"),
    ("APC",   "APC",   "anadarko-petroleum",         "2016-01-01", "2019-08-08"),
    ("DNB",   "DNB",   "dun-bradstreet",             "2016-01-01", "2019-02-08"),
    ("DO",    "DO",    "diamond-offshore-drilling",  "2016-01-01", "2020-04-26"),
    ("FTR",   "FTR",   "frontier-communications",    "2016-01-01", "2020-04-14"),
    ("CBS",   "CBS",   "cbs",                        "2016-01-01", "2019-12-05"),
    ("ENDP",  "ENDP",  "endo-international",         "2016-01-01", "2022-08-16"),
    ("MNK",   "MNK",   "mallinckrodt",               "2016-01-01", "2020-10-12"),

    # ==================================================================
    # GRUPO 3 — Tiingo Free Tier sem dados
    # ==================================================================
    ("BLL",   "BLL",   "ball",                       "2016-01-01", "2022-04-11"),
    ("CDAY",  "CDAY",  "ceridian-hcm",               "2021-09-20", "2023-10-18"),
    ("FRC",   "FRC",   "first-republic-bank",        "2018-07-01", "2023-05-01"),
    ("GPS",   "GPS",   "gap",                        "2016-01-01", "2022-01-10"),
    ("MMC",   "MMC",   "marsh-mclennan",             "2016-01-01", "2025-12-31"),
    ("PEAK",  "PEAK",  "healthpeak-properties",      "2019-07-01", "2024-02-01"),
    ("RE",    "RE",    "everest-re-group",            "2015-07-01", "2023-06-20"),
    ("WRK",   "WRK",   "westrock",                   "2016-01-01", "2024-07-05"),

    # ==================================================================
    # GRUPO 4 — Parciais (arquivo existe mas sem período anterior)
    # ==================================================================
    # FOX/FOXA: arquivos têm Fox Corp (2019+). Queremos 21CF (2016-2019).
    # Macrotrends pode ter histórico 21CF sob o ticker FOX/FOXA.
    # Se falhar, 21CF não está disponível via macrotrends.
    ("FOX",   "FOX",   "fox-corporation",            "2016-01-01", "2019-03-18"),
    ("FOXA",  "FOXA",  "fox-corporation",            "2016-01-01", "2019-03-18"),
    # IR: arquivo tem 2017-05+. Queremos 2016-2017.
    # Old Ingersoll-Rand → split em TT (Trane) + novo IR em fev/2020
    # TT é o sucessor legal do old IR — página TT deve ter histórico completo
    ("IR",    "TT",    "trane-technologies",         "2016-01-01", "2017-05-11"),
]

print(f"Total de targets: {len(targets)}")

Total de targets: 34


In [6]:
results = []

for orig_ticker, mt_ticker, slug, start, end in targets:
    print(f"\n{'─'*55}")
    print(f"[{orig_ticker}] → {mt_ticker}/{slug}  ({start[:7]} → {end[:7]})")

    df = fetch_macrotrends(mt_ticker, slug, start, end)

    if not df.empty:
        out = df.rename(columns={"close": orig_ticker})
        out_path = RECOVERY_DIR / f"{orig_ticker}.csv"
        out.to_csv(out_path, index=False)
        print(f"    ✅ {len(out)} linhas → {out_path.name}")
        results.append({"ticker": orig_ticker, "status": "ok", "rows": len(out),
                        "start": str(df["Date"].min())[:10], "end": str(df["Date"].max())[:10]})
    else:
        print(f"    ❌ Não encontrado")
        results.append({"ticker": orig_ticker, "status": "fail", "rows": 0,
                        "start": "-", "end": "-"})

    time.sleep(3)  # respeitar rate limit


───────────────────────────────────────────────────────
[FB] → META/meta-platforms  (2016-01 → 2021-10)
    macrotrends status: 403
    var originalData não encontrada. HTML: '<!DOCTYPE html><html lang="en-US"><head><title>Just a moment...</title><meta http-equiv="Content-Type" content="text/html; charset=UTF-8"><meta http-equiv="X-UA-Compatible" content="IE=Edge"><meta name="robots" content="noindex,nofollow"><meta name="viewport" content="width=device-width,initial-scal'
    ❌ Não encontrado

───────────────────────────────────────────────────────
[LB] → BBWI/bath-body-works  (2016-01 → 2021-08)
    macrotrends status: 403
    var originalData não encontrada. HTML: '<!DOCTYPE html><html lang="en-US"><head><title>Just a moment...</title><meta http-equiv="Content-Type" content="text/html; charset=UTF-8"><meta http-equiv="X-UA-Compatible" content="IE=Edge"><meta name="robots" content="noindex,nofollow"><meta name="viewport" content="width=device-width,initial-scal'
    ❌ Não encontrado

KeyboardInterrupt: 

In [21]:
df_res = pd.DataFrame(results)
ok   = df_res[df_res.status == "ok"]
fail = df_res[df_res.status != "ok"]

print(f"\n{'='*55}")
print(f"RESUMO DA COLETA")
print(f"{'='*55}")
print(f"✅ Recuperados : {len(ok)} / {len(df_res)}")
print(f"❌ Falharam    : {len(fail)} / {len(df_res)}")

if not ok.empty:
    print("\n--- Recuperados ---")
    print(ok[["ticker", "source", "rows", "start", "end"]].to_string(index=False))

if not fail.empty:
    print("\n--- Falharam ---")
    print(fail[["ticker"]].to_string(index=False))


RESUMO DA COLETA
✅ Recuperados : 0 / 3
❌ Falharam    : 3 / 3

--- Falharam ---
ticker
    FB
    LB
  ANTM


## Incorporar dados recuperados em `prices/`

Execute a célula abaixo **depois de revisar** os arquivos em `macrotrends_recovery/`.

- **Sem arquivo existente** → copia diretamente para `prices/`
- **Arquivo parcial existente** (FOX, FOXA, IR) → faz merge, sem duplicatas, ordenado por data

In [20]:
def incorporate_recovered(orig_ticker: str, dry_run: bool = True):
    rec_file   = RECOVERY_DIR / f"{orig_ticker}.csv"
    price_file = PRICES_DIR   / f"{orig_ticker}.csv"

    if not rec_file.exists():
        print(f"[{orig_ticker}] Sem arquivo em recovery/ — pulando")
        return

    rec_df = pd.read_csv(rec_file, parse_dates=["Date"])

    if price_file.exists():
        existing = pd.read_csv(price_file, parse_dates=["Date"])
        merged = (
            pd.concat([rec_df, existing])
            .drop_duplicates("Date")
            .sort_values("Date")
            .reset_index(drop=True)
        )
        action = f"merge: {len(rec_df)} novas + {len(existing)} existentes = {len(merged)} total"
        if not dry_run:
            merged.to_csv(price_file, index=False)
    else:
        merged = rec_df
        action = f"cópia nova: {len(merged)} linhas"
        if not dry_run:
            shutil.copy(rec_file, price_file)

    status = "[DRY RUN]" if dry_run else "[GRAVADO]"
    print(f"{status} [{orig_ticker}] {action}")


# --- Incorporar apenas os recuperados (dry_run=True para preview) ---
recovered_tickers = [r["ticker"] for r in results if r["status"] == "ok"]
print(f"Tickers prontos para incorporar: {recovered_tickers}\n")

for ticker in recovered_tickers:
    incorporate_recovered(ticker, dry_run=False)   # mude para False para gravar

Tickers prontos para incorporar: []



---
## Parte B — Retry Yahoo / Tiingo para grupos não-reciclados

Tickers que **não** são reciclados — o problema foi erro da API ou limitação do free tier do Tiingo.

| Grupo | Tickers | Estratégia |
|-------|---------|------------|
| API errors | CBS, ENDP, MNK, APC | Tiingo com datas exatas de membership |
| Tiingo free tier (ativos) | BLL, CDAY, GPS, MMC, RE | Yahoo primeiro, Tiingo como fallback |
| Tiingo free tier (renomeados) | ANTM, FBHS, FRC, PEAK, PKI, TMK, WRK | Tiingo com datas exatas |
| Parcial | IR | Yahoo (ticker ainda existe) |

Saída → `macrotrends_recovery/` (mesma pasta de antes). Incorporar com a função da seção anterior.

In [10]:
import yfinance as yf
import requests
import time
import pandas as pd
from pathlib import Path

TIINGO_TOKEN = "dadfd331f2cb44969b8f7468006d20ad62b13262"
PRICES_DIR   = Path("../data_bases/prices")
RECOVERY_DIR = Path("../data_bases/external/macrotrends_recovery")
RECOVERY_DIR.mkdir(parents=True, exist_ok=True)

# (ticker, tiingo_start, tiingo_end, yahoo_ativo)
# tiingo_start/end = janela de membership exata (evita pegar ticker reciclado)
# yahoo_ativo = True → ticker ainda existe no Yahoo com esse nome
targets_b = [
    # --- Grupo 2: API errors / falência / aquisição ---
    ("CBS",  "2015-07-01", "2019-12-05", False),   # merged → ViacomCBS dez/2019
    ("ENDP", "2015-07-01", "2022-08-16", False),   # falência ago/2022
    ("MNK",  "2015-07-01", "2020-10-12", False),   # falência out/2020
    ("APC",  "2015-07-01", "2019-08-08", False),   # adquirida OXY ago/2019
    # --- Grupo 3: ainda ativos com mesmo ticker ---
    ("BLL",  "2015-07-01", "2025-12-31", True),    # Ball Corp, ativo
    ("CDAY", "2021-07-01", "2025-12-31", True),    # Ceridian, ativo
    ("GPS",  "2015-07-01", "2025-12-31", True),    # Gap, ativo
    ("MMC",  "2015-07-01", "2025-12-31", True),    # Marsh McLennan, ativo
    ("RE",   "2015-07-01", "2025-12-31", True),    # Everest Re, ativo
    # --- Grupo 3: renomeados (mesmo ticker até a data de saída) ---
    ("ANTM", "2015-07-01", "2022-06-28", False),   # → ELV jun/2022
    ("FBHS", "2015-07-01", "2022-11-08", False),   # → FBIN nov/2022
    ("FRC",  "2018-07-01", "2023-05-01", False),   # falência mai/2023
    ("PEAK", "2019-07-01", "2024-02-01", False),   # → DOC fev/2024
    ("PKI",  "2015-07-01", "2023-05-04", False),   # → RVTY mar/2023
    ("TMK",  "2015-07-01", "2019-08-08", False),   # → GL ago/2019
    ("WRK",  "2015-07-01", "2024-07-05", False),   # → SW jul/2024
    # --- Parcial: falta 2016-2017 ---
    ("IR",   "2015-07-01", "2017-05-11", True),    # old Ingersoll-Rand (ainda existe como IR)
]

print(f"Targets: {len(targets_b)} tickers")

Targets: 17 tickers


In [11]:
def fetch_yahoo_b(ticker: str, start: str, end: str) -> pd.Series:
    """
    Usa Ticker.history() em vez de download() — evita o bug 'no timezone found'
    de certas versões do yfinance. auto_adjust=False → Close split-only (sem dividendos).
    """
    try:
        t = yf.Ticker(ticker)
        raw = t.history(start=start, end=end, auto_adjust=False)
        if raw.empty or "Close" not in raw.columns:
            return pd.Series(dtype=float, name=ticker)

        close = raw["Close"].copy()
        close.index = pd.to_datetime(close.index).tz_localize(None)
        return close.dropna().rename(ticker)
    except Exception as e:
        print(f"    yahoo error: {e}")
        return pd.Series(dtype=float, name=ticker)


def fetch_tiingo_b(ticker: str, start: str, end: str) -> pd.Series:
    """Mesmo padrão do pipeline_base Parte 2: campo close (sem dividendos)."""
    try:
        r = requests.get(
            f"https://api.tiingo.com/tiingo/daily/{ticker}/prices",
            params={"startDate": start, "endDate": end,
                    "token": TIINGO_TOKEN, "resampleFreq": "daily"},
            timeout=15
        )
        if r.status_code != 200:
            print(f"    tiingo HTTP {r.status_code}")
            return pd.Series(dtype=float, name=ticker)

        data = r.json()
        if not data:
            return pd.Series(dtype=float, name=ticker)

        df = pd.DataFrame(data)
        df["date"] = pd.to_datetime(df["date"]).dt.tz_localize(None)
        return df.set_index("date")["close"].dropna().rename(ticker)
    except Exception as e:
        print(f"    tiingo error: {e}")
        return pd.Series(dtype=float, name=ticker)


def save_recovery(ticker: str, s: pd.Series):
    out = s.reset_index()
    out.columns = ["Date", ticker]
    out.to_csv(RECOVERY_DIR / f"{ticker}.csv", index=False)


print("Funções carregadas.")

Funções carregadas.


In [12]:
results_b = {}

for ticker, t_start, t_end, yahoo_ativo in targets_b:
    print(f"\n[{ticker}]", end="  ")
    s = pd.Series(dtype=float)

    # Passo 1: Yahoo (só para tickers ainda ativos)
    if yahoo_ativo:
        s = fetch_yahoo_b(ticker, t_start, t_end)
        if len(s) > 10:
            save_recovery(ticker, s)
            print(f"✅ Yahoo  {len(s)} linhas  ({s.index[0].date()} → {s.index[-1].date()})")
            results_b[ticker] = "yahoo"
            time.sleep(0.3)
            continue
        else:
            print(f"Yahoo ❌ ({len(s)} linhas) →", end="  ")

    # Passo 2: Tiingo com datas exatas de membership
    s = fetch_tiingo_b(ticker, t_start, t_end)
    if len(s) > 10:
        save_recovery(ticker, s)
        print(f"✅ Tiingo  {len(s)} linhas  ({s.index[0].date()} → {s.index[-1].date()})")
        results_b[ticker] = "tiingo"
    else:
        print(f"Tiingo ❌ ({len(s)} linhas)")
        results_b[ticker] = "fail"

    time.sleep(0.5)

# Resumo
print(f"\n{'='*50}")
ok   = [t for t, v in results_b.items() if v != "fail"]
fail = [t for t, v in results_b.items() if v == "fail"]
print(f"✅ Recuperados ({len(ok)}): {ok}")
print(f"❌ Falharam   ({len(fail)}): {fail}")


[CBS]      tiingo HTTP 404
Tiingo ❌ (0 linhas)

[ENDP]      tiingo HTTP 404
Tiingo ❌ (0 linhas)

[MNK]      tiingo HTTP 404
Tiingo ❌ (0 linhas)

[APC]  Tiingo ❌ (0 linhas)

[BLL]  

$BLL: possibly delisted; no timezone found


Yahoo ❌ (0 linhas) →  Tiingo ❌ (0 linhas)


$CDAY: possibly delisted; no timezone found



[CDAY]  Yahoo ❌ (0 linhas) →      tiingo HTTP 404
Tiingo ❌ (0 linhas)


...


[MMC]  Yahoo ❌ (0 linhas) →  Tiingo ❌ (0 linhas)


$RE: possibly delisted; no timezone found



[RE]  Yahoo ❌ (0 linhas) →  Tiingo ❌ (0 linhas)

[ANTM]  Tiingo ❌ (0 linhas)

[FBHS]  Tiingo ❌ (0 linhas)

[FRC]      tiingo HTTP 404
Tiingo ❌ (0 linhas)

[PEAK]  Tiingo ❌ (0 linhas)

[PKI]  Tiingo ❌ (0 linhas)

[TMK]  Tiingo ❌ (0 linhas)

[WRK]  Tiingo ❌ (1 linhas)


$IR: possibly delisted; no price data found  (1d 2015-07-01 -> 2017-05-11) (Yahoo error = "Data doesn't exist for startDate = 1435723200, endDate = 1494475200")



[IR]  Yahoo ❌ (0 linhas) →  Tiingo ❌ (0 linhas)

✅ Recuperados (0): []
❌ Falharam   (17): ['CBS', 'ENDP', 'MNK', 'APC', 'BLL', 'CDAY', 'GPS', 'MMC', 'RE', 'ANTM', 'FBHS', 'FRC', 'PEAK', 'PKI', 'TMK', 'WRK', 'IR']


---
## Parte C — Financial Data API + Financial Modeling Prep

Dois novos provedores com API key própria.

In [16]:
import requests, time
import pandas as pd
from pathlib import Path

FINANCIALDATA_KEY = "ff6ca5dffb9951ff93400949958695de"
RECOVERY_DIR      = Path("../data_bases/external/macrotrends_recovery")
RECOVERY_DIR.mkdir(parents=True, exist_ok=True)

# Todos os targets não-reciclados (mesmas datas de membership da Parte B)
targets_fd = [
    ("CBS",  "2015-07-01", "2019-12-05"),
    ("ENDP", "2015-07-01", "2022-08-16"),
    ("MNK",  "2015-07-01", "2020-10-12"),
    ("APC",  "2015-07-01", "2019-08-08"),
    ("BLL",  "2015-07-01", "2025-12-31"),
    ("CDAY", "2021-07-01", "2025-12-31"),
    ("GPS",  "2015-07-01", "2025-12-31"),
    ("MMC",  "2015-07-01", "2025-12-31"),
    ("RE",   "2015-07-01", "2025-12-31"),
    ("ANTM", "2015-07-01", "2022-06-28"),
    ("FBHS", "2015-07-01", "2022-11-08"),
    ("FRC",  "2018-07-01", "2023-05-01"),
    ("PEAK", "2019-07-01", "2024-02-01"),
    ("PKI",  "2015-07-01", "2023-05-04"),
    ("TMK",  "2015-07-01", "2019-08-08"),
    ("WRK",  "2015-07-01", "2024-07-05"),
    ("IR",   "2015-07-01", "2017-05-11"),
]


def fetch_financialdata(ticker: str, start: str, end: str) -> pd.Series:
    """
    Busca todos os registros paginando com offset até cobrir o período.
    A API retorna 300 registros por chamada, do mais recente para o mais antigo.
    """
    records = []
    offset  = 0
    start_dt = pd.to_datetime(start)

    while True:
        try:
            r = requests.get(
                "https://financialdata.net/api/v1/stock-prices",
                params={"identifier": ticker, "key": FINANCIALDATA_KEY,
                        "format": "json", "offset": offset},
                timeout=15)
        except Exception as e:
            print(f"    connection error: {e}")
            break

        if not r.ok:
            print(f"    HTTP {r.status_code}")
            break

        page = r.json()
        if not isinstance(page, list) or len(page) == 0:
            break  # sem mais dados

        records.extend(page)

        # Data mais antiga desta página
        oldest = pd.to_datetime(page[-1]["date"])
        if oldest <= start_dt:
            break  # chegamos ao período que precisamos

        offset += 300
        time.sleep(0.4)

    if not records:
        return pd.Series(dtype=float, name=ticker)

    df = pd.DataFrame(records)
    df["date"] = pd.to_datetime(df["date"])
    df = df[(df["date"] >= start) & (df["date"] <= end)]
    df = df.sort_values("date").drop_duplicates("date")

    return df.set_index("date")["close"].rename(ticker)


# ── Coleta ──
results_fd = {}

for ticker, t_start, t_end in targets_fd:
    print(f"[{ticker}]  ", end="", flush=True)
    s = fetch_financialdata(ticker, t_start, t_end)

    if len(s) > 10:
        out = s.reset_index()
        out.columns = ["Date", ticker]
        out.to_csv(RECOVERY_DIR / f"{ticker}.csv", index=False)
        print(f"✅  {len(s)} linhas  ({s.index[0].date()} → {s.index[-1].date()})")
        results_fd[ticker] = "ok"
    else:
        print(f"❌  {len(s)} linhas — sem dados")
        results_fd[ticker] = "fail"

    time.sleep(0.5)

print(f"\n{'='*50}")
ok   = [t for t, v in results_fd.items() if v == "ok"]
fail = [t for t, v in results_fd.items() if v == "fail"]
print(f"✅ Recuperados ({len(ok)}): {ok}")
print(f"❌ Falharam   ({len(fail)}): {fail}")

[CBS]  ❌  0 linhas — sem dados
[ENDP]  ❌  0 linhas — sem dados
[MNK]  ❌  0 linhas — sem dados
[APC]  ❌  0 linhas — sem dados
[BLL]  ❌  0 linhas — sem dados
[CDAY]  ❌  0 linhas — sem dados
[GPS]  ❌  0 linhas — sem dados
[MMC]  ✅  2596 linhas  (2015-09-04 → 2025-12-31)
[RE]  ❌  0 linhas — sem dados
[ANTM]  ❌  0 linhas — sem dados
[FBHS]  ❌  0 linhas — sem dados
[FRC]  ❌  0 linhas — sem dados
[PEAK]  ❌  0 linhas — sem dados
[PKI]  ❌  0 linhas — sem dados
[TMK]  ❌  0 linhas — sem dados
[WRK]  ❌  0 linhas — sem dados
[IR]  ❌  0 linhas — sem dados

✅ Recuperados (1): ['MMC']
❌ Falharam   (16): ['CBS', 'ENDP', 'MNK', 'APC', 'BLL', 'CDAY', 'GPS', 'RE', 'ANTM', 'FBHS', 'FRC', 'PEAK', 'PKI', 'TMK', 'WRK', 'IR']


In [17]:
# ── Validação: Financial Data `close` vs Yahoo Close existente ──
# Busca AAPL nas duas fontes e calcula ratio (deve ser ≈ 1.0 se o padrão for igual)

import pandas as pd, requests, time
from pathlib import Path

FINANCIALDATA_KEY = "ff6ca5dffb9951ff93400949958695de"
PRICES_DIR = Path("../data_bases/prices")

# 1) AAPL do Financial Data API — pegar duas páginas para ter 2023-2024
records_aapl = []
for offset in [300, 600]:   # offset 300 ≈ 2024-2025, offset 600 ≈ 2023-2024
    r = requests.get("https://financialdata.net/api/v1/stock-prices",
                     params={"identifier": "AAPL", "key": FINANCIALDATA_KEY,
                             "format": "json", "offset": offset}, timeout=15)
    records_aapl.extend(r.json())
    time.sleep(0.4)

fd_aapl = pd.DataFrame(records_aapl)
fd_aapl["date"] = pd.to_datetime(fd_aapl["date"])
fd_aapl = fd_aapl.set_index("date")["close"].sort_index()
print(f"Financial Data AAPL: {len(fd_aapl)} registros | {fd_aapl.index[0].date()} → {fd_aapl.index[-1].date()}")
print(f"  Exemplos: {fd_aapl.iloc[:3].to_dict()}")

# 2) AAPL do arquivo existente (Yahoo)
yahoo_aapl = pd.read_csv(PRICES_DIR / "AAPL.csv", index_col=0, parse_dates=True)
yahoo_aapl.index = pd.to_datetime(yahoo_aapl.index).tz_localize(None)
yahoo_aapl = yahoo_aapl.iloc[:, 0].sort_index()
print(f"\nYahoo AAPL (existente): {len(yahoo_aapl)} registros | {yahoo_aapl.index[0].date()} → {yahoo_aapl.index[-1].date()}")

# 3) Comparar datas em comum
common = fd_aapl.index.intersection(yahoo_aapl.index)
cmp = pd.DataFrame({"fd": fd_aapl[common], "yahoo": yahoo_aapl[common]}).dropna()
cmp["ratio"] = cmp["fd"] / cmp["yahoo"]

print(f"\n── Comparação em {len(cmp)} datas em comum ──")
print(cmp.head(5).to_string())
print(f"\nRatio: mean={cmp['ratio'].mean():.6f}  std={cmp['ratio'].std():.6f}  "
      f"min={cmp['ratio'].min():.4f}  max={cmp['ratio'].max():.4f}")
if cmp["ratio"].std() < 0.001:
    print("✅ Financial Data close = Yahoo Close (mesmo padrão — split-adjusted, sem dividendos)")
else:
    print("⚠️  Valores divergem — verificar se é adjClose ou close diferente")

Financial Data AAPL: 600 registros | 2022-09-26 → 2025-02-14
  Exemplos: {Timestamp('2022-09-26 00:00:00'): 150.77, Timestamp('2022-09-27 00:00:00'): 151.76, Timestamp('2022-09-28 00:00:00'): 149.84}

Yahoo AAPL (existente): 2641 registros | 2015-07-01 → 2025-12-30

── Comparação em 600 datas em comum ──
                fd       yahoo  ratio
2022-09-26  150.77  150.770004    1.0
2022-09-27  151.76  151.759995    1.0
2022-09-28  149.84  149.839996    1.0
2022-09-29  142.48  142.479996    1.0
2022-09-30  138.20  138.199997    1.0

Ratio: mean=1.000000  std=0.000002  min=1.0000  max=1.0000
✅ Financial Data close = Yahoo Close (mesmo padrão — split-adjusted, sem dividendos)


In [22]:
import shutil, pandas as pd
from pathlib import Path

PRICES_DIR   = Path("../data_bases/prices")
RECOVERY_DIR = Path("../data_bases/external/macrotrends_recovery")

def incorporate_recovered(ticker: str, dry_run: bool = True):
    rec_file   = RECOVERY_DIR / f"{ticker}.csv"
    price_file = PRICES_DIR   / f"{ticker}.csv"

    if not rec_file.exists():
        print(f"[{ticker}] Sem arquivo em recovery/")
        return

    rec_df = pd.read_csv(rec_file, parse_dates=["Date"])

    if price_file.exists():
        existing = pd.read_csv(price_file, parse_dates=["Date"])
        merged = (pd.concat([rec_df, existing])
                  .drop_duplicates("Date")
                  .sort_values("Date")
                  .reset_index(drop=True))
        action = f"merge: {len(rec_df)} novas + {len(existing)} existentes = {len(merged)} total"
        if not dry_run:
            merged.to_csv(price_file, index=False)
    else:
        merged = rec_df
        action = f"cópia nova: {len(merged)} linhas"
        if not dry_run:
            shutil.copy(rec_file, price_file)

    status = "[DRY RUN]" if dry_run else "[GRAVADO]"
    print(f"{status} [{ticker}] {action}")
    if not dry_run:
        print(f"  Verificação: {len(pd.read_csv(price_file))} linhas em prices/{ticker}.csv")

# Preview primeiro
incorporate_recovered("MMC", dry_run=False)

# Descomentar para gravar:
# incorporate_recovered("MMC", dry_run=False)

[GRAVADO] [MMC] cópia nova: 2596 linhas
  Verificação: 2596 linhas em prices/MMC.csv
